In [1]:
# ============================================
# CELL 1: Setup Path & Load Data
# ============================================
from pathlib import Path
import pandas as pd
import numpy as np

try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(data_processed / "makassar_features_spatial.parquet")

print("DATA:", df.shape)


# ============================================
# CELL 2: Add Controlled Noise (REALISTIC)
# ============================================
np.random.seed(42)

df["rain_noise"] = np.random.normal(0, 3, len(df))
df["rainfall_real"] = (df["rainfall"] + df["rain_noise"]).clip(lower=0)

print("✔ Noise added")


# ============================================
# CELL 3: Create Lag Features
# ============================================
df = df.sort_values(["grid_id", "date"])

lags = [1, 3, 7, 14]

for lag in lags:
    df[f"rain_real_lag_{lag}"] = df.groupby("grid_id")["rainfall_real"].shift(lag)

df["rain_real_7d_sum"] = (
    df.groupby("grid_id")["rainfall_real"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

df["rain_real_30d_sum"] = (
    df.groupby("grid_id")["rainfall_real"]
    .rolling(30)
    .sum()
    .reset_index(level=0, drop=True)
)

print("✔ Lag features created")


# ============================================
# CELL 4: PHYSICS-BASED FLOOD SCORE (FIXED)
# ============================================
score = (
    0.6 * df["rain_real_7d_sum"] +   # dominan (hydrology)
    0.25 * df["rain_real_lag_1"] +
    0.1 * df["rain_real_lag_3"] +
    0.05 * (df["spatial_factor"] * 100)
)

# noise kecil → realism tanpa merusak signal
score = score + np.random.normal(0, 5, len(df))

print("✔ Flood score computed")


# ============================================
# CELL 5: Sigmoid → Probability (STRONGER)
# ============================================
prob = 1 / (1 + np.exp(-0.03 * (score - 170)))

print("✔ Probability generated")


# ============================================
# CELL 6: Generate Flood Label
# ============================================
df["flood_label_v2"] = (np.random.rand(len(df)) < prob).astype(int)

print("✔ Initial flood label created")


# ============================================
# CELL 7: CONTROL FLOOD RATIO
# ============================================
target_ratio = 0.08

current_ratio = df["flood_label_v2"].mean()

if current_ratio > target_ratio:
    drop_idx = df[df["flood_label_v2"] == 1].sample(
        frac=1 - (target_ratio / current_ratio),
        random_state=42
    ).index
    
    df.loc[drop_idx, "flood_label_v2"] = 0

print("✔ Flood ratio adjusted")
print("Flood ratio:", df["flood_label_v2"].mean())


# ============================================
# CELL 8: Select Columns & Clean Data
# ============================================
cols_needed = [
    "grid_id", "date",
    "rainfall_real",
    "rain_real_lag_1",
    "rain_real_lag_3",
    "rain_real_lag_7",
    "rain_real_lag_14",
    "rain_real_7d_sum",
    "rain_real_30d_sum",
    "month",
    "spatial_factor",
    "flood_label_v2"
]

df_model = df[cols_needed].dropna()

print("\n=== SHAPE AFTER CLEANING ===")
print(df_model.shape)


# ============================================
# CELL 9: SAVE FINAL DATASET
# ============================================
out_path = data_processed / "makassar_final_dataset.parquet"

df_model.to_parquet(out_path, index=False)

print("✔ FINAL DATASET SAVED:", out_path)


# ============================================
# CELL 10: FULL Q1 DEBUG CHECK
# ============================================

print("\n=== DATASET INFO ===")
print("Shape:", df_model.shape)

print("\n=== FLOOD RATIO ===")
print(df_model["flood_label_v2"].mean())

print("\n=== FLOOD DISTRIBUTION ===")
print(df_model["flood_label_v2"].value_counts(normalize=True))

print("\n=== FLOOD BY MONTH ===")
print(df_model.groupby("month")["flood_label_v2"].mean())

print("\n=== RAINFALL vs FLOOD ===")
print(df_model.groupby("flood_label_v2")["rainfall_real"].describe())

print("\n=== LAG FEATURE STATS ===")
lag_cols = [c for c in df_model.columns if "lag" in c or "sum" in c]
print(df_model[lag_cols].describe())

print("\n=== SPATIAL FACTOR ===")
print(df_model["spatial_factor"].describe())

print("\n=== CORRELATION WITH FLOOD ===")
numeric_cols = df_model.select_dtypes(include=["float64", "int64"]).columns
corr = df_model[numeric_cols].corr()["flood_label_v2"].sort_values(ascending=False)
print(corr)

print("\n=== EXTREME CASE CHECK ===")
print("Max rainfall:", df_model["rainfall_real"].max())
print("Mean rainfall:", df_model["rainfall_real"].mean())

print("\n=== SAMPLE DATA ===")
print(df_model.sample(10, random_state=42))

print("\n✔ 01C FINAL (Q1 READY)")

DATA: (5004844, 14)
✔ Noise added
✔ Lag features created
✔ Flood score computed
✔ Probability generated
✔ Initial flood label created
✔ Flood ratio adjusted
Flood ratio: 0.08000009590708522

=== SHAPE AFTER CLEANING ===
(4922658, 12)
✔ FINAL DATASET SAVED: d:\GITHUB\Project\flood-ml-research\data\processed\makassar_final_dataset.parquet

=== DATASET INFO ===
Shape: (4922658, 12)

=== FLOOD RATIO ===
0.08026618952606498

=== FLOOD DISTRIBUTION ===
flood_label_v2
0    0.919734
1    0.080266
Name: proportion, dtype: float64

=== FLOOD BY MONTH ===
month
1     0.080455
2     0.079883
3     0.080014
4     0.080962
5     0.079436
6     0.080318
7     0.080270
8     0.079703
9     0.081157
10    0.080620
11    0.080275
12    0.080040
Name: flood_label_v2, dtype: float64

=== RAINFALL vs FLOOD ===
                    count       mean        std  min        25%        50%  \
flood_label_v2                                                               
0               4527535.0  19.647972  14.36